# 2. Base Meteorológica (NASA POWER, 2020–2024) — *Final_datos_objetivo_1.ipynb*

## ¿Qué se construyó y para qué sirve?
Se construyó una **base meteorológica limpia y consistente** (a resolución **diaria** y **mensual**) asociada a las **estaciones** del sistema de calidad del aire.  
Su propósito es aportar **covariables de ajuste** (temperatura, humedad, viento y precipitación) que ayudan a:

- **Controlar confusión** en modelos de salud (la meteorología afecta tanto las concentraciones de contaminantes como la ocurrencia de eventos respiratorios).
- **Alinear escalas temporales** con los paneles de contaminantes (que terminan agregándose a mensual) y con salud (RIPS/mortalidad típicamente mensual).
- **Garantizar trazabilidad y control de calidad**, al dejar explícita la fuente, cobertura y estructura del panel.

---

## Insumos (crudos) y estructura de carpetas
**Ubicación crudo**
- Carpeta: `BASES DE DATOS/BASE DE DATOS METEREOLOGICA/CRUDO/`
- Archivo: `nasa_power_daily.csv`

**Estructura cruda típica**
- `date, T2M, RH2M, WS2M, PRECTOTCORR, estacion, localidad, lat, lon`

> Nota: el archivo venía separado por comas y con líneas entrecomilladas; por eso se implementó una lectura robusta.

**Salida (limpios)**
- Carpeta: `BASES DE DATOS/BASE DE DATOS METEREOLOGICA/LIMPIOS/`

---

## ¿Qué se hizo en el notebook? (paso a paso y por qué)

### 1) Definición de rutas (paths) y control de existencia
**Qué se hizo**
- Se definieron rutas relativas desde `ROOT` para ubicar `CRUDO` y crear/usar `LIMPIOS`.

**Por qué**
- Evita depender de rutas “copiadas a mano” (que fallan por espacios, tildes o cambios mínimos).
- Hace que el pipeline sea **reproducible** dentro del proyecto.

**Qué aporta**
- Menos errores de ejecución y más orden para mantener el proyecto a lo largo del tiempo.

---

### 2) Lectura robusta del CSV (formato con comillas / comas)
**Qué se hizo**
- Se implementó una función que:
  - lee el archivo como texto,
  - remueve comillas externas por línea (si existen),
  - separa por comas,
  - arma el DataFrame con encabezados consistentes.

**Por qué**
- Algunos CSV exportados o ensamblados desde APIs llegan en formatos “irregulares” para `pd.read_csv()` estándar.
- Asegura que las columnas queden exactamente como se espera.

**Qué aporta**
- Lectura estable y repetible del crudo sin necesidad de editar manualmente el archivo.

---

### 3) Limpieza de texto (corrección de codificación)
**Qué se hizo**
- Se corrigieron caracteres mal decodificados (ej.: `EngativÃ¡` → `Engativá`) en nombres de localidades.

**Por qué**
- Las inconsistencias de codificación rompen:
  - cruces posteriores,
  - agregaciones por localidad,
  - visualizaciones y documentación.

**Qué aporta**
- Nombres consistentes (importante cuando la base crece y se integra con salud/demografía).

---

### 4) Tipificación, estandarización de variables y creación del panel diario
**Qué se hizo**
- Se convirtió `date` a `Fecha` tipo datetime.
- Variables meteorológicas a numérico:
  - `T2M` → `Temp_media`
  - `RH2M` → `HR`
  - `WS2M` → `Vel_viento`
  - `PRECTOTCORR` → `Precipitación`
- Se añadieron campos auxiliares:
  - `Año`, `Mes`
  - `Fuente = NASA_POWER`
  - `lat`, `lon` (para trazabilidad espacial)
- Se construyó `meteo_diario` con estructura estándar:

`Fecha | Estacion | Localidad | Temp_media | HR | Vel_viento | Presión | Precipitación | Fuente | Año | Mes | lat | lon`

**Por qué**
- Los modelos (y los cruces) necesitan variables con nombres claros y tipos correctos.
- `Año` y `Mes` facilitan agregación y unión con paneles mensuales (contaminantes/salud).
- Lat/Lon ayudan a validar que cada serie corresponde a la estación correcta.

**Qué aporta**
- Base diaria coherente, lista para análisis exploratorio o agregación.

> **Presión**: en esta extracción NASA no estaba disponible, por eso se dejó como `NaN` y se documentó con cobertura 0.0.

---

### 5) Guardado del panel diario en LIMPIOS
**Qué se hizo**
- Se exportó:
  - `panel_meteo_diario_nasa_2020_2024.csv`

**Por qué**
- Permite que el resto del proyecto consuma una versión “congelada” y consistente sin re-procesar cada vez.

**Qué aporta**
- Trazabilidad del dato y modularidad del pipeline (cada base queda lista como insumo).

---

### 6) Construcción del panel mensual y métricas de cobertura
**Qué se hizo**
- Se agregaron los datos diarios a **mensual por estación**, calculando:
  - `Temp_media` (promedio mensual)
  - `HR` (promedio mensual)
  - `Vel_viento` (promedio mensual)
  - `Precipitación` (acumulado mensual) + `Precipitación_media` (promedio)
  - `dias_esperados` (días del mes)
- Se construyó cobertura mensual por variable:
  - `Cobertura_Temp = n_dias_con_dato / dias_esperados` (análogo para HR, viento, precip)
  - `Cobertura_Presion` queda 0.0 porque no hay datos de presión en NASA

**Por qué**
- La cobertura es una medida simple pero crucial para:
  - evaluar completitud,
  - justificar exclusiones o banderas,
  - soportar análisis de sensibilidad (por ejemplo, umbrales de cobertura).

**Qué aporta**
- Control de calidad explícito (no “supuesto”).
- Insumo directo para la unión con contaminantes y, luego, salud.

**Archivos generados**
- `panel_meteo_mensual_nasa_2020_2024.csv`
- `cobertura_meteo_mensual_nasa_2020_2024.csv`

---

### 7) Validación final (checks automáticos)
**Qué se hizo**
- Se verificó:
  - rango temporal completo: **2020-01-01 → 2024-12-31**
  - número de estaciones: **17**
  - número de localidades: **11**
  - meses únicos: **60** (2020–2024 completo)
  - meses por estación: **60/60**
  - cobertura mensual: 1.0 en Temp/HR/Viento/Precip; 0.0 en Presión

**Por qué**
- Evita avanzar con una base “incompleta” sin darse cuenta.
- Deja evidencia cuantitativa para documentar en el capítulo de datos.

**Qué aporta**
- Confianza en que la base meteorológica no introduce huecos temporales que luego confundan resultados.

---

## Productos finales del Apartado 2 (lo que queda “listo” para el proyecto)
**(1) Diario limpio**
- `.../LIMPIOS/panel_meteo_diario_nasa_2020_2024.csv`

**(2) Mensual limpio**
- `.../LIMPIOS/panel_meteo_mensual_nasa_2020_2024.csv`

**(3) Cobertura mensual**
- `.../LIMPIOS/cobertura_meteo_mensual_nasa_2020_2024.csv`

---

## ¿Cómo se usa después dentro del pipeline?
- El **panel mensual meteorológico** se une con **contaminantes mensuales** por:
  - `Año + Mes + Estacion`
- Ese panel ambiental integrado será la base para:
  - modelar asociaciones exposición–salud,
  - análisis de sensibilidad (p. ej. filtrando por cobertura),
  - alimentar escenarios y evaluaciones con BenMAP-CE (según cómo estructures las entradas).

---


In [1]:
import re
import calendar
from pathlib import Path

import pandas as pd
import numpy as np

# =========================
# PATHS (Notebook dentro de /CODIGO)
# =========================
ROOT = Path.cwd().parent  # ...\TRABAJO DE GRADO BEN-MAP
print("ROOT:", ROOT)

bases_dir = ROOT / "BASES DE DATOS"
if not bases_dir.exists():
    raise FileNotFoundError(f"No existe: {bases_dir}")

meteo_dir = bases_dir / "BASE DE DATOS METEREOLOGICA"
if not meteo_dir.exists():
    raise FileNotFoundError(f"No existe: {meteo_dir}")

CRUDO_DIR = meteo_dir / "CRUDO"
LIMPIOS_DIR = meteo_dir / "LIMPIOS"
LIMPIOS_DIR.mkdir(parents=True, exist_ok=True)

# archivo NASA (si cambia el nombre, lo busca)
nasa_path = CRUDO_DIR / "nasa_power_daily.csv"
if not nasa_path.exists():
    cands = list(CRUDO_DIR.glob("*nasa*power*.csv"))
    if not cands:
        raise FileNotFoundError(f"No encontré nasa_power_daily.csv ni archivos *nasa*power*.csv en: {CRUDO_DIR}")
    nasa_path = cands[0]

print("CRUDO_DIR:", CRUDO_DIR)
print("LIMPIOS_DIR:", LIMPIOS_DIR)
print("NASA FILE:", nasa_path)


ROOT: d:\TRABAJO DE GRADO BEN-MAP
CRUDO_DIR: d:\TRABAJO DE GRADO BEN-MAP\BASES DE DATOS\BASE DE DATOS METEREOLOGICA\CRUDO
LIMPIOS_DIR: d:\TRABAJO DE GRADO BEN-MAP\BASES DE DATOS\BASE DE DATOS METEREOLOGICA\LIMPIOS
NASA FILE: d:\TRABAJO DE GRADO BEN-MAP\BASES DE DATOS\BASE DE DATOS METEREOLOGICA\CRUDO\nasa_power_daily.csv


In [5]:
import calendar

def fix_mojibake(x: str) -> str:
    """
    Arregla textos tipo 'EngativÃ¡' -> 'Engativá' (mojibake típico).
    """
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return x
    s = str(x).strip()
    if ("Ã" in s) or ("Â" in s):
        try:
            return s.encode("latin1").decode("utf-8")
        except Exception:
            return s
    return s


def read_nasa_power_daily(file_path: Path) -> pd.DataFrame:
    """
    Lee CSV de NASA POWER cuando:
    - la primera línea (header) viene entre comillas
    - cada fila también viene entre comillas
    Ej:
    "date,T2M,RH2M,..."
    "2020-01-01,18.87,85.65,..."
    """
    lines = file_path.read_text(encoding="utf-8-sig").splitlines()
    rows = []
    for ln in lines:
        ln = ln.strip()
        if ln.startswith('"') and ln.endswith('"'):
            ln = ln[1:-1]
        rows.append(ln.split(","))

    header = [c.strip() for c in rows[0]]
    data = rows[1:]
    df = pd.DataFrame(data, columns=header)
    return df


def days_in_month(year: int, month: int) -> int:
    return calendar.monthrange(int(year), int(month))[1]


## carga el archivo NASA, valida columnas, corrige texto raro y te deja un dataframe diario estandarizado (meteo_diario) + un preview para revisar que todo quedó bien.

In [6]:
# === Cargar NASA POWER (crudo) ===
raw = read_nasa_power_daily(nasa_path)

print("Columnas detectadas:", list(raw.columns))
print("Filas/Cols (raw):", raw.shape)

# Normalizar nombres (strip)
raw.columns = [c.strip() for c in raw.columns]

# Validar columnas mínimas esperadas (según tu ejemplo)
expected = {"date","T2M","RH2M","WS2M","PRECTOTCORR","estacion","localidad","lat","lon"}
missing = expected - set(raw.columns)
if missing:
    raise ValueError(
        f"Faltan columnas en NASA: {missing}\n"
        f"Columnas detectadas: {list(raw.columns)}"
    )

df = raw.copy()

# Tipos numéricos
for c in ["T2M","RH2M","WS2M","PRECTOTCORR","lat","lon"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# Fecha
df["Fecha"] = pd.to_datetime(df["date"], errors="coerce")

# Texto limpio
df["Estacion"] = df["estacion"].astype(str).str.strip()
df["Localidad"] = df["localidad"].astype(str).map(fix_mojibake).str.strip()

# Variables estandarizadas
df["Temp_media"] = df["T2M"]
df["HR"] = df["RH2M"]
df["Vel_viento"] = df["WS2M"]
df["Precipitación"] = df["PRECTOTCORR"]

# NASA POWER diario no trae presión -> dejamos NaN por ahora
df["Presión"] = np.nan

df["Fuente"] = "NASA_POWER"
df["Año"] = df["Fecha"].dt.year
df["Mes"] = df["Fecha"].dt.month

# Base diaria limpia (con lat/lon para trazabilidad)
meteo_diario = df[[
    "Fecha","Estacion","Localidad",
    "Temp_media","HR","Vel_viento","Presión","Precipitación",
    "Fuente","Año","Mes","lat","lon"
]].dropna(subset=["Fecha","Estacion"])

# Orden y duplicados
meteo_diario = meteo_diario.drop_duplicates(subset=["Fecha","Estacion","Fuente"]).sort_values(["Fecha","Estacion"])

print("✅ meteo_diario:", meteo_diario.shape)
meteo_diario.head()


Columnas detectadas: ['date', 'T2M', 'RH2M', 'WS2M', 'PRECTOTCORR', 'estacion', 'localidad', 'lat', 'lon']
Filas/Cols (raw): (31059, 9)
✅ meteo_diario: (31059, 13)


,Fecha,Estacion,Localidad,Temp_media,HR,Vel_viento,Presión,Precipitación,Fuente,Año,Mes,lat,lon
0,2020-01-01,Bolivia,Engativá,18.87,85.65,1.02,NaN,1.67,NASA_POWER,2020,1,4.736978,-74.555000
1827,2020-01-01,Carvajal - Sevillana,Tunjuelito,18.87,85.65,1.02,NaN,1.67,NASA_POWER,2020,1,4.589722,-74.217500
3654,2020-01-01,Centro de Alto Rendimiento,Barrios Unidos,18.87,85.65,1.02,NaN,1.67,NASA_POWER,2020,1,4.658556,-74.088556
5481,2020-01-01,Ciudad Bolivar,Ciudad Bolívar,18.87,85.65,1.02,NaN,1.67,NASA_POWER,2020,1,4.577778,-74.165167
7308,2020-01-01,Colina,Suba,18.87,85.65,1.02,NaN,1.67,NASA_POWER,2020,1,4.737194,-74.069472


In [7]:
# === Guardar base diaria limpia ===
out_diario = LIMPIOS_DIR / "panel_meteo_diario_nasa_2020_2024.csv"
meteo_diario.to_csv(out_diario, index=False, encoding="utf-8-sig")

print("✅ Guardado:", out_diario)
print("Tamaño (filas, cols):", meteo_diario.shape)
print("NaN por columna:\n", meteo_diario.isna().sum())


✅ Guardado: d:\TRABAJO DE GRADO BEN-MAP\BASES DE DATOS\BASE DE DATOS METEREOLOGICA\LIMPIOS\panel_meteo_diario_nasa_2020_2024.csv
Tamaño (filas, cols): (31059, 13)
NaN por columna:
 Fecha                0
Estacion             0
Localidad            0
Temp_media           0
HR                   0
Vel_viento           0
Presión          31059
Precipitación        0
Fuente               0
Año                  0
Mes                  0
lat                  0
lon                  0
dtype: int64


## Celda para verificar coincidencia con contaminantes

In [13]:
# Buscar automáticamente el resultados_contaminantes_mensual.csv (sin depender de espacios)
cont_root = bases_dir  # ya existe en tu celda 1 de meteo

cont_folders = [p for p in cont_root.iterdir() if p.is_dir() and "CONTAMINANTES POR ESTACION" in p.name.upper()]
print("Candidatos:", [p.name for p in cont_folders])

if not cont_folders:
    raise FileNotFoundError("No encontré carpeta que contenga 'CONTAMINANTES POR ESTACION' en BASES DE DATOS.")

# buscar el archivo dentro de cualquier candidato
cont_path = None
for p in cont_folders:
    candidate = p / "TRATADO" / "resultados_contaminantes_mensual.csv"
    if candidate.exists():
        cont_path = candidate
        break

if cont_path is None:
    # fallback: búsqueda recursiva por si cambia nombre de carpeta interna
    hits = []
    for p in cont_folders:
        hits.extend(list(p.rglob("resultados_contaminantes_mensual.csv")))
    if hits:
        cont_path = hits[0]

if cont_path is None:
    raise FileNotFoundError("No encontré 'resultados_contaminantes_mensual.csv' dentro de las carpetas candidatas.")

print("✅ Encontrado:", cont_path)

cont = pd.read_csv(cont_path)
print("OK contaminantes:", cont.shape)
cont.head()

Candidatos: ['BASE DE DATOS DE CONTAMINANTES POR ESTACION  (2020-2024)']
✅ Encontrado: d:\TRABAJO DE GRADO BEN-MAP\BASES DE DATOS\BASE DE DATOS DE CONTAMINANTES POR ESTACION  (2020-2024)\TRATADO\resultados_contaminantes_mensual.csv
OK contaminantes: (862, 9)


,Año,Mes,Estacion,NO2,O3,PM25,Cobertura_NO2,Cobertura_O3,Cobertura_PM25
0,2020,1,Carvajal - Sevillana,3.291089,NaN,29.811594,0.950269,NaN,0.927419
1,2020,1,Centro de Alto Rendimiento,13.822458,17.690691,13.672938,0.951613,0.952957,0.928763
2,2020,1,Fontibon,16.696307,15.080595,19.646154,0.946237,0.948925,0.943548
3,2020,1,Guaymaral,11.346686,13.813510,14.772134,0.912634,0.915323,0.926075
4,2020,1,Kennedy,17.867042,21.176695,23.648442,0.954301,0.951613,0.948925


In [14]:
est_cont = set(cont["Estacion"].astype(str).str.strip().unique())
est_meteo = set(meteo_diario["Estacion"].astype(str).str.strip().unique())

solo_en_cont = sorted(list(est_cont - est_meteo))
solo_en_meteo = sorted(list(est_meteo - est_cont))
inter = sorted(list(est_cont & est_meteo))

print("Estaciones contaminantes:", len(est_cont))
print("Estaciones meteo (NASA):", len(est_meteo))
print("Coinciden (intersección):", len(inter))
print("Solo en contaminantes:", len(solo_en_cont))
print("Solo en meteo:", len(solo_en_meteo))

print("\nEjemplos solo en contaminantes:", solo_en_cont[:15])
print("Ejemplos solo en meteo:", solo_en_meteo[:15])


Estaciones contaminantes: 16
Estaciones meteo (NASA): 17
Coinciden (intersección): 16
Solo en contaminantes: 0
Solo en meteo: 1

Ejemplos solo en contaminantes: []
Ejemplos solo en meteo: ['SUBA']


In [ ]:
# === Panel mensual + cobertura mensual (NASA) ===

# 1) días esperados por mes
ym = meteo_diario[["Año","Mes"]].drop_duplicates().copy()
ym["dias_esperados"] = ym.apply(lambda r: days_in_month(r["Año"], r["Mes"]), axis=1)

dfm = meteo_diario.merge(ym, on=["Año","Mes"], how="left")

grp = ["Año","Mes","Estacion","Localidad","Fuente"]

# 2) Panel mensual (promedios) + precipitación acumulada mensual
meteo_mensual = dfm.groupby(grp).agg(
    Temp_media=("Temp_media","mean"),
    HR=("HR","mean"),
    Vel_viento=("Vel_viento","mean"),
    Presión=("Presión","mean"),              # quedará NaN mientras no integremos IDEAM/RMCAB
    Precipitación=("Precipitación","sum"),   # acumulado mensual
    Precipitación_media=("Precipitación","mean"),
    lat=("lat","first"),
    lon=("lon","first"),
    dias_esperados=("dias_esperados","first"),
).reset_index()

# 3) Cobertura mensual por variable (conteo días no nulos / días esperados)
cov = dfm.groupby(grp).agg(
    dias_esperados=("dias_esperados","first"),
    n_Temp=("Temp_media","count"),
    n_HR=("HR","count"),
    n_Viento=("Vel_viento","count"),
    n_Presion=("Presión","count"),
    n_Precip=("Precipitación","count"),
).reset_index()

cov["Cobertura_Temp"]    = cov["n_Temp"]    / cov["dias_esperados"]
cov["Cobertura_HR"]      = cov["n_HR"]      / cov["dias_esperados"]
cov["Cobertura_Viento"]  = cov["n_Viento"]  / cov["dias_esperados"]
cov["Cobertura_Presion"] = cov["n_Presion"] / cov["dias_esperados"]
cov["Cobertura_Precip"]  = cov["n_Precip"]  / cov["dias_esperados"]

cobertura_mensual = cov[[
    "Año","Mes","Estacion","Localidad","Fuente",
    "Cobertura_Temp","Cobertura_HR","Cobertura_Viento","Cobertura_Presion","Cobertura_Precip",
    "dias_esperados","n_Temp","n_HR","n_Viento","n_Presion","n_Precip"
]].copy()

# 4) Guardar
out_mensual = LIMPIOS_DIR / "panel_meteo_mensual_nasa_2020_2024.csv"
out_cov     = LIMPIOS_DIR / "cobertura_meteo_mensual_nasa_2020_2024.c   sv"

meteo_mensual.to_csv(out_mensual, index=False, encoding="utf-8-sig")
cobertura_mensual.to_csv(out_cov, index=False, encoding="utf-8-sig")

print("✅ meteo_mensual:", meteo_mensual.shape, "->", out_mensual)
print("✅ cobertura_mensual:", cobertura_mensual.shape, "->", out_cov)

# 5) chequeo rápido
print("Cobertura Temp min/max:", cobertura_mensual["Cobertura_Temp"].min(), cobertura_mensual["Cobertura_Temp"].max())
print("Cobertura Precip min/max:", cobertura_mensual["Cobertura_Precip"].min(), cobertura_mensual["Cobertura_Precip"].max())


✅ meteo_mensual: (1020, 14) -> d:\TRABAJO DE GRADO BEN-MAP\BASES DE DATOS\BASE DE DATOS METEREOLOGICA\LIMPIOS\panel_meteo_mensual_nasa_2020_2024.csv
✅ cobertura_mensual: (1020, 16) -> d:\TRABAJO DE GRADO BEN-MAP\BASES DE DATOS\BASE DE DATOS METEREOLOGICA\LIMPIOS\cobertura_meteo_mensual_nasa_2020_2024.csv
Cobertura Temp min/max: 1.0 1.0
Cobertura Precip min/max: 1.0 1.0


## unir y guardar panel ambiental mensual

In [16]:
# === PANEL AMBIENTAL MENSUAL (contaminantes + meteo + coberturas) ===

# 1) Encontrar contaminantes mensual (preferimos _con_flags si existe)
cont_folders = [p for p in bases_dir.iterdir() if p.is_dir() and "CONTAMINANTES POR ESTACION" in p.name.upper()]
if not cont_folders:
    raise FileNotFoundError("No encontré carpeta 'CONTAMINANTES POR ESTACION' en BASES DE DATOS.")

cand_flags, cand_base = None, None
for p in cont_folders:
    f1 = p / "TRATADO" / "resultados_contaminantes_mensual_con_flags.csv"
    f2 = p / "TRATADO" / "resultados_contaminantes_mensual.csv"
    if f1.exists():
        cand_flags = f1
    if f2.exists():
        cand_base = f2

cont_path = cand_flags if cand_flags is not None else cand_base
if cont_path is None:
    raise FileNotFoundError("No encontré resultados_contaminantes_mensual*.csv dentro de TRATADO.")

cont = pd.read_csv(cont_path)
cont["Estacion"] = cont["Estacion"].astype(str).str.strip()

print("✅ Contaminantes:", cont.shape, "->", cont_path)

# 2) Cargar meteo mensual
meteo_path = LIMPIOS_DIR / "panel_meteo_mensual_nasa_2020_2024.csv"
if not meteo_path.exists():
    raise FileNotFoundError(f"No existe: {meteo_path}")

meteo = pd.read_csv(meteo_path)
meteo["Estacion"] = meteo["Estacion"].astype(str).str.strip()

print("✅ Meteo mensual:", meteo.shape, "->", meteo_path)

# 3) (Opcional) cobertura meteo mensual
cov_path = LIMPIOS_DIR / "cobertura_meteo_mensual_nasa_2020_2024.csv"
cov = None
if cov_path.exists():
    cov = pd.read_csv(cov_path)
    cov["Estacion"] = cov["Estacion"].astype(str).str.strip()
    print("✅ Cobertura meteo:", cov.shape, "->", cov_path)

# 4) Unir cont + meteo
panel = cont.merge(
    meteo,
    on=["Año", "Mes", "Estacion"],
    how="left",
    suffixes=("", "_meteo")
)

# 5) Unir cobertura meteo (si está)
if cov is not None:
    panel = panel.merge(
        cov[["Año","Mes","Estacion","Cobertura_Temp","Cobertura_HR","Cobertura_Viento","Cobertura_Presion","Cobertura_Precip"]],
        on=["Año","Mes","Estacion"],
        how="left"
    )

print("✅ Panel ambiental mensual:", panel.shape)

# 6) Diagnóstico rápido: cuántos quedaron sin meteo
faltan_meteo = panel["Temp_media"].isna().sum()
print("Filas sin meteo (Temp_media NaN):", faltan_meteo)

# 7) Guardar en una carpeta nueva (limpia y clara)
OUT_PANEL_DIR = bases_dir / "PANEL AMBIENTAL INTEGRADO"
OUT_PANEL_DIR.mkdir(parents=True, exist_ok=True)

out_panel = OUT_PANEL_DIR / "panel_ambiental_mensual_2020_2024.csv"
panel.to_csv(out_panel, index=False, encoding="utf-8-sig")
print("✅ Guardado:", out_panel)

panel.head()


✅ Contaminantes: (862, 12) -> d:\TRABAJO DE GRADO BEN-MAP\BASES DE DATOS\BASE DE DATOS DE CONTAMINANTES POR ESTACION  (2020-2024)\TRATADO\resultados_contaminantes_mensual_con_flags.csv
✅ Meteo mensual: (1020, 14) -> d:\TRABAJO DE GRADO BEN-MAP\BASES DE DATOS\BASE DE DATOS METEREOLOGICA\LIMPIOS\panel_meteo_mensual_nasa_2020_2024.csv
✅ Cobertura meteo: (1020, 16) -> d:\TRABAJO DE GRADO BEN-MAP\BASES DE DATOS\BASE DE DATOS METEREOLOGICA\LIMPIOS\cobertura_meteo_mensual_nasa_2020_2024.csv
✅ Panel ambiental mensual: (862, 28)
Filas sin meteo (Temp_media NaN): 0
✅ Guardado: d:\TRABAJO DE GRADO BEN-MAP\BASES DE DATOS\PANEL AMBIENTAL INTEGRADO\panel_ambiental_mensual_2020_2024.csv


,Año,Mes,Estacion,NO2,O3,PM25,Cobertura_NO2,Cobertura_O3,Cobertura_PM25,valido_NO2,...,Precipitación,Precipitación_media,lat,lon,dias_esperados,Cobertura_Temp,Cobertura_HR,Cobertura_Viento,Cobertura_Presion,Cobertura_Precip
0,2020,1,Carvajal - Sevillana,3.291089,NaN,29.811594,0.950269,NaN,0.927419,True,...,23.51,0.758387,4.589722,-74.217500,31,1.0,1.0,1.0,0.0,1.0
1,2020,1,Centro de Alto Rendimiento,13.822458,17.690691,13.672938,0.951613,0.952957,0.928763,True,...,23.51,0.758387,4.658556,-74.088556,31,1.0,1.0,1.0,0.0,1.0
2,2020,1,Fontibon,16.696307,15.080595,19.646154,0.946237,0.948925,0.943548,True,...,23.51,0.758387,4.679722,-74.142500,31,1.0,1.0,1.0,0.0,1.0
3,2020,1,Guaymaral,11.346686,13.813510,14.772134,0.912634,0.915323,0.926075,True,...,29.07,0.937742,4.783756,-74.044183,31,1.0,1.0,1.0,0.0,1.0
4,2020,1,Kennedy,17.867042,21.176695,23.648442,0.954301,0.951613,0.948925,True,...,23.51,0.758387,4.621389,-74.140833,31,1.0,1.0,1.0,0.0,1.0



### Meteorología (NASA) — OK

* **Diario**: `panel_meteo_diario_nasa_2020_2024.csv` ✅
* **Mensual**: `panel_meteo_mensual_nasa_2020_2024.csv` ✅
* **Cobertura mensual**: `cobertura_meteo_mensual_nasa_2020_2024.csv` ✅

  * Cobertura 1.0 en todo (perfecto, NASA está completo)
  * `Cobertura_Presion = 0.0` porque **no existe presión** en la fuente NASA (correcto)

### Integración ambiental mensual — OK

* `panel_ambiental_mensual_2020_2024.csv` ✅
* Filas sin meteo = **0** ✅ (o sea, el join quedó perfecto)
* Columnas finales (28) con contaminantes + meteo + coberturas + flags ✅

Eso significa que **ya tienes el insumo ambiental mensual listo** para:

* modelos de salud (RIPS/mortalidad mensual)
* sensibilidad (lags, índices, etc.)
* BenMAP como base de exposición mensual si así lo estás planteando

---

“La variable presión no estuvo disponible en la extracción NASA POWER utilizada; por tanto, se conservó como faltante y su cobertura mensual fue 0. En futuros refinamientos se integrará presión observada de IDEAM/RMCAB si se dispone.”

## rango de fechas, # estaciones/localidades, conteos esperados vs reales, meses esperados (60), y detecta si falta algún mes o estación.

In [17]:
# === CHECK FINAL BASE METEOROLÓGICA (NASA) ===
import pandas as pd
import numpy as np

# 1) Rango temporal y cardinalidades (DIARIO)
print("=== DIARIO ===")
print("Archivo diario:", (LIMPIOS_DIR / "panel_meteo_diario_nasa_2020_2024.csv"))
print("Rango fechas:", meteo_diario["Fecha"].min(), "->", meteo_diario["Fecha"].max())
print("# Estaciones:", meteo_diario["Estacion"].nunique())
print("# Localidades:", meteo_diario["Localidad"].nunique())
print("Filas diario:", len(meteo_diario))

# 2) Esperado teórico (aprox) = sum(días por año) * estaciones
# (solo como referencia, porque estaciones podrían variar en el tiempo)
stations = meteo_diario["Estacion"].nunique()
days_total = (meteo_diario["Fecha"].max() - meteo_diario["Fecha"].min()).days + 1
print("Referencia esperado aprox (días*estaciones):", days_total * stations)

# 3) Mensual: filas esperadas y meses completos
print("\n=== MENSUAL ===")
meteo_mensual = pd.read_csv(LIMPIOS_DIR / "panel_meteo_mensual_nasa_2020_2024.csv")
print("Filas mensual:", meteo_mensual.shape)

# meses únicos
meses = meteo_mensual[["Año","Mes"]].drop_duplicates().sort_values(["Año","Mes"])
print("# Meses únicos:", len(meses), "(esperado 60 si es 2020-2024 completo)")
print("Primer mes:", meses.iloc[0].to_dict(), "| Último mes:", meses.iloc[-1].to_dict())

# 4) Verificar que cada estación tenga los 60 meses
conteo_meses_est = meteo_mensual.groupby("Estacion")[["Año","Mes"]].apply(lambda x: x.drop_duplicates().shape[0])
print("\nMeses por estación (min/max):", conteo_meses_est.min(), conteo_meses_est.max())

faltantes_est = conteo_meses_est[conteo_meses_est < len(meses)]
if len(faltantes_est) == 0:
    print("✅ Todas las estaciones tienen todos los meses disponibles.")
else:
    print("⚠️ Estaciones con meses faltantes:")
    display(faltantes_est.sort_values())

# 5) Cobertura mensual (si existe)
cov_path = LIMPIOS_DIR / "cobertura_meteo_mensual_nasa_2020_2024.csv"
if cov_path.exists():
    cov = pd.read_csv(cov_path)
    print("\n=== COBERTURA MENSUAL ===")
    print("Cobertura Temp (min/max):", cov["Cobertura_Temp"].min(), cov["Cobertura_Temp"].max())
    print("Cobertura HR (min/max):", cov["Cobertura_HR"].min(), cov["Cobertura_HR"].max())
    print("Cobertura Viento (min/max):", cov["Cobertura_Viento"].min(), cov["Cobertura_Viento"].max())
    print("Cobertura Precip (min/max):", cov["Cobertura_Precip"].min(), cov["Cobertura_Precip"].max())
    # presión probablemente 0 si no existe
    if "Cobertura_Presion" in cov.columns:
        print("Cobertura Presión (min/max):", cov["Cobertura_Presion"].min(), cov["Cobertura_Presion"].max())
else:
    print("\n(No se encontró archivo de cobertura mensual. Si lo generaste, revisa nombre/ruta.)")


=== DIARIO ===
Archivo diario: d:\TRABAJO DE GRADO BEN-MAP\BASES DE DATOS\BASE DE DATOS METEREOLOGICA\LIMPIOS\panel_meteo_diario_nasa_2020_2024.csv
Rango fechas: 2020-01-01 00:00:00 -> 2024-12-31 00:00:00
# Estaciones: 17
# Localidades: 11
Filas diario: 31059
Referencia esperado aprox (días*estaciones): 31059

=== MENSUAL ===
Filas mensual: (1020, 14)
# Meses únicos: 60 (esperado 60 si es 2020-2024 completo)
Primer mes: {'Año': 2020, 'Mes': 1} | Último mes: {'Año': 2024, 'Mes': 12}

Meses por estación (min/max): 60 60
✅ Todas las estaciones tienen todos los meses disponibles.

=== COBERTURA MENSUAL ===
Cobertura Temp (min/max): 1.0 1.0
Cobertura HR (min/max): 1.0 1.0
Cobertura Viento (min/max): 1.0 1.0
Cobertura Precip (min/max): 1.0 1.0
Cobertura Presión (min/max): 0.0 0.0
